In [1]:
import json
from pathlib import Path

DATA_DIR = Path("data") / "processed"

with (DATA_DIR / "train.json").open(encoding="utf-8") as file:
    train_documents = json.load(file)

with (DATA_DIR / "validation.json").open(encoding="utf-8") as file:
    valid_documents = json.load(file)

with (DATA_DIR / "vocab.json").open(encoding="utf-8") as file:
    vocab_list = json.load(file)

In [2]:
token_to_id = {
    token: index
    for index, token in enumerate(vocab_list)
}

id_to_token = vocab_list

print("Sözlük boyutu:", len(vocab_list))
print("'the' kimliği:", token_to_id["the"])
print("'<unk>' kimliği:", token_to_id["<unk>"])

Sözlük boyutu: 20000
'the' kimliği: 17872
'<unk>' kimliği: 754


In [3]:
train_ids = [
    [token_to_id[token] for token in document["tokens"]]
    for document in train_documents
]

valid_ids = [
    [token_to_id[token] for token in document["tokens"]]
    for document in valid_documents
]

print("İlk token'lar:", train_documents[0]["tokens"][:10])
print("Kimlikleri:", train_ids[0][:10])

decoded = [id_to_token[index] for index in train_ids[0][:10]]

assert decoded == train_documents[0]["tokens"][:10]
print("Geri dönüşüm kontrolü geçti.")

İlk token'lar: ['produced', 'by', 'susan', 'skinner', ',', 'chris', '<unk>', ',', '<unk>', 'patten']
Kimlikleri: [14109, 3375, 17499, 16425, 11, 3900, 754, 11, 754, 13174]
Geri dönüşüm kontrolü geçti.


In [4]:
import importlib
import lm.lstm as lstm

importlib.reload(lstm)

SEQ_LEN = 32

train_inputs, train_targets = lstm.create_sequences(
    train_ids,
    bos_id=token_to_id["<bos>"],
    eos_id=token_to_id["<eos>"],
    sequence_length=SEQ_LEN,
)

valid_inputs, valid_targets = lstm.create_sequences(
    valid_ids,
    bos_id=token_to_id["<bos>"],
    eos_id=token_to_id["<eos>"],
    sequence_length=SEQ_LEN,
)

print("Train örnek sayısı:", len(train_inputs))
print("Validation örnek sayısı:", len(valid_inputs))

Train örnek sayısı: 31258
Validation örnek sayısı: 3126


In [5]:
print(
    "Girdi:",
    [id_to_token[index] for index in train_inputs[0][:8]],
)

print(
    "Hedef:",
    [id_to_token[index] for index in train_targets[0][:8]],
)

assert all(
    len(x) == len(y)
    for x, y in zip(train_inputs, train_targets)
)

expected_predictions = sum(len(ids) + 1 for ids in train_ids)

assert sum(len(y) for y in train_targets) == expected_predictions

print("Toplam eğitim tahmini:", expected_predictions)

Girdi: ['<bos>', 'produced', 'by', 'susan', 'skinner', ',', 'chris', '<unk>']
Hedef: ['produced', 'by', 'susan', 'skinner', ',', 'chris', '<unk>', ',']
Toplam eğitim tahmini: 1000020


In [6]:
import torch

print("PyTorch:", torch.__version__)
print("GPU erişimi:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.ones(1, device="cuda")
    print("GPU hesaplama testi:", (x + x).item())

PyTorch: 2.15.0+cu132
GPU erişimi: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
GPU hesaplama testi: 2.0


In [7]:
from functools import partial
from torch.utils.data import DataLoader
from config import SEED

importlib.reload(lstm)

BATCH_SIZE = 32

collate_fn = partial(
    lstm.collate_batch,
    input_pad_id=token_to_id["<eos>"],
)

train_loader = DataLoader(
    list(zip(train_inputs, train_targets)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)

valid_loader = DataLoader(
    list(zip(valid_inputs, valid_targets)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

print("Train batch sayısı:", len(train_loader))
print("Validation batch sayısı:", len(valid_loader))

Train batch sayısı: 977
Validation batch sayısı: 98


In [8]:
device = torch.device("cuda")

x_batch, y_batch = next(iter(train_loader))

x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

print("Girdi:", x_batch.shape, x_batch.device)
print("Hedef:", y_batch.shape, y_batch.device)

assert x_batch.shape == y_batch.shape

Girdi: torch.Size([32, 32]) cuda:0
Hedef: torch.Size([32, 32]) cuda:0


In [9]:
importlib.reload(lstm)

torch.manual_seed(SEED)

model = lstm.LSTMLanguageModel(
    vocab_size=len(vocab_list),
    bos_id=token_to_id["<bos>"],
    embedding_dim=128,
    hidden_dim=128,
).to(device)

print(model)
print("Model cihazı:", next(model.parameters()).device)

LSTMLanguageModel(
  (embedding): Embedding(20000, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (output): Linear(in_features=128, out_features=20000, bias=True)
)
Model cihazı: cuda:0


In [10]:
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=-100
)

model.zero_grad(set_to_none=True)

logits, _ = model(x_batch)

print("Çıktı boyutu:", logits.shape)

loss = criterion(
    logits.reshape(-1, len(vocab_list)),
    y_batch.reshape(-1),
)

assert torch.isfinite(loss).item()

print("Başlangıç loss:", loss.item())

loss.backward()

assert all(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all().item()
    for parameter in model.parameters()
)

print("Geri yayılım kontrolü geçti.")

model.zero_grad(set_to_none=True)

Çıktı boyutu: torch.Size([32, 32, 20000])
Başlangıç loss: 9.897747039794922
Geri yayılım kontrolü geçti.


In [11]:
import math
import time
import json
from pathlib import Path

importlib.reload(lstm)

EPOCHS = 10

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
)

best_val_loss = float("inf")
history = []

ARTIFACT_DIR = Path("artifacts")
RESULT_DIR = Path("results")

ARTIFACT_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

checkpoint_path = ARTIFACT_DIR / "lstm_best.pt"

In [12]:
for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()

    train_loss = lstm.run_epoch(
        model, train_loader, device, optimizer
    )

    train_seconds = time.perf_counter() - started

    val_loss = lstm.run_epoch(
        model, valid_loader, device
    )

    train_ppl = math.exp(train_loss)
    val_ppl = math.exp(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "train_perplexity": train_ppl,
        "validation_perplexity": val_ppl,
        "train_seconds": train_seconds,
    })

    print(
        f"\nEpoch {epoch}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f} | "
        f"Validation PPL: {val_ppl:.2f} | "
        f"Eğitim: {train_seconds:.1f} saniye"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "model_config": {
                "vocab_size": model.embedding.num_embeddings,
                "bos_id": model.bos_id,
                "embedding_dim": model.embedding.embedding_dim,
                "hidden_dim": model.lstm.hidden_size,
            },
            "vocab_list": vocab_list,
            "sequence_length": SEQ_LEN,
            "seed": SEED,
            "validation_loss": val_loss,
            "validation_perplexity": val_ppl,
        }, checkpoint_path)

        print("En iyi model kaydedildi.")

    with (RESULT_DIR / "lstm_history.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(history, file, indent=2)

Batch 100/977 | loss: 7.4788
Batch 200/977 | loss: 7.0240
Batch 300/977 | loss: 6.8116
Batch 400/977 | loss: 6.6666
Batch 500/977 | loss: 6.5654
Batch 600/977 | loss: 6.4809
Batch 700/977 | loss: 6.4113
Batch 800/977 | loss: 6.3521
Batch 900/977 | loss: 6.3005

Epoch 1/10 | Train loss: 6.2661 | Validation loss: 5.6363 | Validation PPL: 280.44 | Eğitim: 10.1 saniye
En iyi model kaydedildi.
Batch 100/977 | loss: 5.7641
Batch 200/977 | loss: 5.7482
Batch 300/977 | loss: 5.7292
Batch 400/977 | loss: 5.7143
Batch 500/977 | loss: 5.7011
Batch 600/977 | loss: 5.6871
Batch 700/977 | loss: 5.6743
Batch 800/977 | loss: 5.6641
Batch 900/977 | loss: 5.6540

Epoch 2/10 | Train loss: 5.6464 | Validation loss: 5.4516 | Validation PPL: 233.14 | Eğitim: 9.3 saniye
En iyi model kaydedildi.
Batch 100/977 | loss: 5.4494
Batch 200/977 | loss: 5.4494
Batch 300/977 | loss: 5.4466
Batch 400/977 | loss: 5.4406
Batch 500/977 | loss: 5.4361
Batch 600/977 | loss: 5.4290
Batch 700/977 | loss: 5.4227
Batch 800/977 

In [13]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Seçilen epoch:", checkpoint["epoch"])
print(
    "En iyi validation perplexity:",
    checkpoint["validation_perplexity"],
)

Seçilen epoch: 7
En iyi validation perplexity: 200.4935487271036


In [14]:
import lm.generation as generation

importlib.reload(generation)

text = generation.generate_lstm_text(
    model=model,
    prompt="it was",
    token_to_id=token_to_id,
    id_to_token=id_to_token,
    max_new_tokens=30,
    temperature=1.0,
    top_k=10,
    seed=42,
)

print(text)

it was a very man, but she could hardly have had been in a time he, and was the great of which he saw him, but, and he
